# Model B — Embedding mixup

Cross-entropy with beta(0.4, 0.4) mixup. Kaggle public LB: 0.739.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import os
import sys
from pathlib import Path

def _load_project():
    my_drive = Path("/content/drive/MyDrive")
    init_candidates = [
        p for p in my_drive.rglob("colab_init.py")
        if (p.parent / "birdclef").is_dir()
    ]
    if init_candidates:
        init_path = min(init_candidates, key=lambda p: len(p.parts))
        spec = importlib.util.spec_from_file_location("colab_init", init_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return
    roots = [
        nb.parent.resolve()
        for nb in my_drive.rglob("02_download_and_extract_embeddings.ipynb")
        if (nb.parent / "birdclef").is_dir()
    ]
    if not roots:
        raise FileNotFoundError(
            "Unzip the full repository on Google Drive, then open a notebook from that folder."
        )
    root = min(roots, key=lambda p: len(p.parts))
    sys.path.insert(0, str(root))
    os.chdir(root)

_load_project()

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=False)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.0 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df, NUM_CLASSES, _ = prepare_baseline_data()
model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)


In [4]:
SAVE_DIR = MODELS_DIR / "mixup_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 10
mixup_fold_scores = []

for TRAIN_FOLD in range(5):
    best_auc = 0.0
    train_df = df[df["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df[df["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=False)
    valid_ds = BirdDataset(valid_df, is_tta=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            lam = np.random.beta(0.4, 0.4)
            batch_size = inputs.size()[0]
            index = torch.randperm(batch_size).to(device)
            mixed_inputs = lam * inputs + (1 - lam) * inputs[index]
            optimizer.zero_grad()
            outputs = model(mixed_inputs)
            loss = lam * criterion(outputs, labels) + (1 - lam) * criterion(outputs, labels[index])

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        if current_auc > best_auc:
            best_auc = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    onnx_save_path = os.path.join(str(SAVE_DIR), f"bird_moe_fold{TRAIN_FOLD}.onnx")
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, 1536).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported ONNX model to {onnx_save_path}")
    mixup_fold_scores.append(best_auc)

print(f"CV score: {np.mean(mixup_fold_scores):.4f} (+/- {np.std(mixup_fold_scores):.4f})")


100%|██████████| 7110/7110 [00:01<00:00, 3891.87it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.8907 | Val Loss: 0.8218 | Val AUC: 0.9914
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 1.3760 | Val Loss: 0.7937 | Val AUC: 0.9910
Epoch 03/10 | Train Loss: 1.1693 | Val Loss: 0.8066 | Val AUC: 0.9913
Epoch 04/10 | Train Loss: 1.1411 | Val Loss: 0.8145 | Val AUC: 0.9908
Epoch 05/10 | Train Loss: 1.0967 | Val Loss: 0.8141 | Val AUC: 0.9907
Epoch 06/10 | Train Loss: 1.0375 | Val Loss: 0.8122 | Val AUC: 0.9884
Epoch 07/10 | Train Loss: 1.0001 | Val Loss: 0.8197 | Val AUC: 0.9891
Epoch 08/10 | Train Loss: 1.0340 | Val Loss: 0.8230 | Val AUC: 0.9889
Epoch 09/10 | Train Loss: 1.0538 | Val Loss: 0.8234 | Val AUC: 0.9885
Epoch 10/10 | Train Loss: 1.0085 | Val Loss: 0.8241 | Val AUC: 0.9885
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/learning_curves_fold0.png
E

100%|██████████| 7110/7110 [00:01<00:00, 4139.43it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.4973 | Val Loss: 0.6553 | Val AUC: 0.9963
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 1.2401 | Val Loss: 0.6748 | Val AUC: 0.9959
Epoch 03/10 | Train Loss: 1.1768 | Val Loss: 0.6985 | Val AUC: 0.9950
Epoch 04/10 | Train Loss: 1.1115 | Val Loss: 0.7142 | Val AUC: 0.9938
Epoch 05/10 | Train Loss: 1.0596 | Val Loss: 0.7448 | Val AUC: 0.9921
Epoch 06/10 | Train Loss: 1.0553 | Val Loss: 0.7534 | Val AUC: 0.9920
Epoch 07/10 | Train Loss: 1.0671 | Val Loss: 0.7532 | Val AUC: 0.9914
Epoch 08/10 | Train Loss: 1.0122 | Val Loss: 0.7667 | Val AUC: 0.9902
Epoch 09/10 | Train Loss: 1.0136 | Val Loss: 0.7734 | Val AUC: 0.9889
Epoch 10/10 | Train Loss: 1.0181 | Val Loss: 0.7727 | Val AUC: 0.9907
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/learning_curves_fold1.png
E

100%|██████████| 7110/7110 [00:02<00:00, 3420.74it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.3132 | Val Loss: 0.4738 | Val AUC: 0.9987
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 1.2102 | Val Loss: 0.5199 | Val AUC: 0.9977
Epoch 03/10 | Train Loss: 1.1195 | Val Loss: 0.5534 | Val AUC: 0.9969
Epoch 04/10 | Train Loss: 1.0925 | Val Loss: 0.5901 | Val AUC: 0.9961
Epoch 05/10 | Train Loss: 1.1027 | Val Loss: 0.6289 | Val AUC: 0.9955
Epoch 06/10 | Train Loss: 1.0061 | Val Loss: 0.6310 | Val AUC: 0.9933
Epoch 07/10 | Train Loss: 1.0130 | Val Loss: 0.6596 | Val AUC: 0.9924
Epoch 08/10 | Train Loss: 1.0308 | Val Loss: 0.6740 | Val AUC: 0.9929
Epoch 09/10 | Train Loss: 1.0510 | Val Loss: 0.6828 | Val AUC: 0.9919
Epoch 10/10 | Train Loss: 0.9602 | Val Loss: 0.6847 | Val AUC: 0.9917
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/learning_curves_fold2.png
E

100%|██████████| 7110/7110 [00:01<00:00, 4628.44it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.2181 | Val Loss: 0.3846 | Val AUC: 0.9994
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 1.1439 | Val Loss: 0.4200 | Val AUC: 0.9992
Epoch 03/10 | Train Loss: 1.1004 | Val Loss: 0.4748 | Val AUC: 0.9987
Epoch 04/10 | Train Loss: 1.1053 | Val Loss: 0.5348 | Val AUC: 0.9982
Epoch 05/10 | Train Loss: 1.0302 | Val Loss: 0.5592 | Val AUC: 0.9976
Epoch 06/10 | Train Loss: 1.0279 | Val Loss: 0.5791 | Val AUC: 0.9973
Epoch 07/10 | Train Loss: 1.0235 | Val Loss: 0.5964 | Val AUC: 0.9968
Epoch 08/10 | Train Loss: 0.9758 | Val Loss: 0.6336 | Val AUC: 0.9964
Epoch 09/10 | Train Loss: 0.9699 | Val Loss: 0.6311 | Val AUC: 0.9957
Epoch 10/10 | Train Loss: 0.9845 | Val Loss: 0.6412 | Val AUC: 0.9953
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/learning_curves_fold3.png
E

100%|██████████| 7109/7109 [00:01<00:00, 4159.04it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 1.2433 | Val Loss: 0.3110 | Val AUC: 0.9996
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 1.1085 | Val Loss: 0.3692 | Val AUC: 0.9994
Epoch 03/10 | Train Loss: 1.0479 | Val Loss: 0.4383 | Val AUC: 0.9990
Epoch 04/10 | Train Loss: 1.0511 | Val Loss: 0.4722 | Val AUC: 0.9986
Epoch 05/10 | Train Loss: 1.0006 | Val Loss: 0.5030 | Val AUC: 0.9984
Epoch 06/10 | Train Loss: 0.9444 | Val Loss: 0.5271 | Val AUC: 0.9980
Epoch 07/10 | Train Loss: 1.0161 | Val Loss: 0.5654 | Val AUC: 0.9971
Epoch 08/10 | Train Loss: 0.9680 | Val Loss: 0.5778 | Val AUC: 0.9965
Epoch 09/10 | Train Loss: 1.0019 | Val Loss: 0.6036 | Val AUC: 0.9964
Epoch 10/10 | Train Loss: 0.9672 | Val Loss: 0.6250 | Val AUC: 0.9961
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/learning_curves_fold4.png
E